# Valeria: fine-tune Gemma 4 E2B (text + photos)

Runtime → Change runtime type → **GPU** (T4 works; L4/A100 is faster). Run the cells top to bottom.

What it does:
1. builds the synthetic text dataset;
2. downloads and prepares the CC BY 4.0 wound/sprain photo datasets from Roboflow;
3. scores the **base** model (baseline);
4. trains run 1 (text only) and run 2 (text + photos), scoring each one;
5. saves the results to Google Drive.

In [ ]:
!nvidia-smi

## 1. Get the code
If the GitHub repo is private, either add a GitHub token, or upload a zip of the repo to Drive and unzip it here instead.

In [ ]:
!git clone https://github.com/tommy-Samy123/valeria.git
%cd valeria

In [ ]:
!pip install -q -r training/requirements-train.txt -r training/requirements-data.txt

## 2. Text dataset

In [ ]:
!python training/scripts/build_text_dataset.py

## 3. Photo dataset
Add your Roboflow API key as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar). Never paste it into a cell.

In [ ]:
import os
from google.colab import userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
!python training/scripts/download_images.py
!python training/scripts/prepare_images.py --max-per-class 600
!cat training/data/images/stats.json

## 4. Smoke test (about 2 minutes): checks that loading, collation and saving all work

In [ ]:
!python training/scripts/train_gemma4.py --max-steps 10 \
  --train training/data/text/train.jsonl training/data/images/train.jsonl \
  --val training/data/text/val.jsonl --val-limit 20 --out training/outputs/smoke

## 5. Baseline: base model, no fine-tuning

In [ ]:
!python training/scripts/evaluate.py --model unsloth/gemma-4-E2B-it --limit 150 \
  --data training/data/text/test.jsonl training/data/images/test.jsonl --out training/outputs/eval_base

## 6. Run 1: text only

In [ ]:
!python training/scripts/train_gemma4.py \
  --train training/data/text/train.jsonl --val training/data/text/val.jsonl --out training/outputs/text_r16
!python training/scripts/evaluate.py --model training/outputs/text_r16/lora \
  --data training/data/text/test.jsonl training/data/images/test.jsonl --out training/outputs/eval_text_r16

## 7. Run 2: text + photos (saves a merged model for export)

In [ ]:
!python training/scripts/train_gemma4.py --merge \
  --train training/data/text/train.jsonl training/data/images/train.jsonl \
  --val training/data/text/val.jsonl training/data/images/val.jsonl --out training/outputs/mix_r16
!python training/scripts/evaluate.py --model training/outputs/mix_r16/lora \
  --data training/data/text/test.jsonl training/data/images/test.jsonl --out training/outputs/eval_mix_r16

## 8. Save results to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/valeria_runs
!cp -r training/outputs/eval_* training/outputs/text_r16/lora training/outputs/mix_r16 /content/drive/MyDrive/valeria_runs/